In [3]:
import pandas as pd
from pyliftover import LiftOver

manifest_path = "/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins/test_code/valid/valid_mqtl/infinium-methylationepic-v-1-0-b5-manifest-file.csv"  # 已解压的 CSV 路径
# 跳过前 7 行注释，根据实际文件可微调
epict_manifest = pd.read_csv(
    manifest_path,
    skiprows=7,
    usecols=["IlmnID", "CHR", "MAPINFO"]
)
# 重命名以标准化列名
epict_manifest.columns = ["cpg_id", "chr_hg19", "pos_hg19"]
epict_manifest.set_index("cpg_id", inplace=True)

In [4]:
lo = LiftOver("hg19", "hg38")  # 默认从 hg19 -> hg38

In [33]:
# df = pd.read_csv("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/GTEX/BreastMammaryTissue.regular.perm.fdr.txt",
#                         sep='\t')
df = pd.read_csv("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/GTEX/Ovary.regular.perm.fdr.txt",
                        sep='\t')
df = df.join(epict_manifest, on="cpg_id", how="left")  # 左连接，缺失值为 NaN

def liftover_pos(row):
    if pd.isna(row["chr_hg19"]) or pd.isna(row["pos_hg19"]):
        return pd.Series([None, None])
    chrom = f"chr{row['chr_hg19']}"
    # MAPINFO 是 1-based, pyliftover 也期望 1-based
    result = lo.convert_coordinate(chrom, int(row["pos_hg19"]))
    if not result:
        return pd.Series([None, None])
    # 取第一个匹配，result 返回 [(chrom, pos, strand, score), ...]
    new_chrom, new_pos, strand, score = result[0]
    return pd.Series([new_chrom.replace("chr", ""), new_pos])
# 执行批量 liftover
df[['chr_hg38', 'pos_hg38']] = df.apply(liftover_pos, axis=1)

In [6]:
df.shape

(754054, 12)

In [35]:
df.shape

(754054, 12)

In [42]:
maf_thresh     = 0.05  # 最低次等位基因频率
abs_slope_min  = 0.5   # 最小绝对效应值
slope_se_max   = 0.05   # 最大效应标准误
p_nominal_max  = 1e-5  # 最大名义p值
p_permuted_max = 0.05  # 最大置换p值
fdr_thresh     = 0.05  # 最大 FDR

filtered = df[
    (df["maf"] >= maf_thresh) &
    (df["slope"].abs() >= abs_slope_min) &
    (df["slope_se"] <= slope_se_max) &
    (df["pval_nominal"] <= p_nominal_max) &
    (df["pval_permuted"] <= p_permuted_max) &
    (df["qval"] <= fdr_thresh)
].copy()

print(f"筛选后保留 {len(filtered)} 条 mQTL 记录")

筛选后保留 1290 条 mQTL 记录


In [41]:
filtered.head()

,cpg_id,variant_id,maf,slope,slope_se,pval_nominal,pval_permuted,qval,chr_hg19,pos_hg19,chr_hg38,pos_hg38,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,chrom,CPG_region_start,CPG_region_end,effect_size
0,cg20485607,chr1_119665674_T_C_b38,0.360714,-1.21932,0.031727,5.778350e-66,9.787140e-60,4.304529e-54,1,120217696.0,1,119675073,119665674,119665674,T,C,chr1,119675073,119675073,-1.21932
1,cg06987384,chr12_56594558_C_A_b38,0.475000,1.18511,0.031037,1.128040e-65,1.249120e-58,2.154462e-53,12,56988451.0,12,56594667,56594558,56594558,C,A,chr12,56594667,56594667,1.18511
2,cg05765582,chr2_190811874_G_A_b38,0.439286,1.12245,0.029933,7.301790e-65,1.469570e-58,2.154462e-53,2,191677683.0,2,190812957,190811874,190811874,G,A,chr2,190812957,190812957,1.12245
3,cg16982741,chr19_38825554_C_T_b38,0.464286,-1.14154,0.032370,3.940210e-62,6.390840e-58,7.026966e-53,19,39323018.0,19,38832378,38825554,38825554,C,T,chr19,38832378,38832378,-1.14154
4,cg16307866,chr4_7127829_G_A_b38,0.307143,1.26729,0.036369,1.337270e-61,2.160130e-57,1.900114e-52,4,7129517.0,4,7127790,7127829,7127829,G,A,chr4,7127790,7127790,1.26729


In [38]:
filtered['SNP_region_start'] = filtered['variant_id'].str.split('_').str[1].astype(int)
filtered['SNP_region_end'] = filtered['variant_id'].str.split('_').str[1].astype(int)
filtered['SNP_ref'] = filtered['variant_id'].str.split('_').str[2]
filtered['SNP_alt'] = filtered['variant_id'].str.split('_').str[3]
filtered['chrom'] = filtered['variant_id'].str.split('_').str[0]
filtered['CPG_region_start'] = filtered['pos_hg38']
filtered['CPG_region_end'] = filtered['pos_hg38']
filtered['effect_size'] = filtered['slope']

In [39]:
filtered_save = filtered.copy()
filtered_save = filtered_save[['chrom', 'SNP_region_start', 'SNP_region_end', 'SNP_ref', 'SNP_alt', 'CPG_region_start', 'CPG_region_end', 'effect_size']]

In [40]:
filtered_save.head()

,chrom,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,CPG_region_start,CPG_region_end,effect_size
0,chr1,119665674,119665674,T,C,119675073,119675073,-1.21932
1,chr12,56594558,56594558,C,A,56594667,56594667,1.18511
2,chr2,190811874,190811874,G,A,190812957,190812957,1.12245
3,chr19,38825554,38825554,C,T,38832378,38832378,-1.14154
4,chr4,7127829,7127829,G,A,7127790,7127790,1.26729


In [28]:
import os 
if not os.path.exists(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/GTEX/"):
    os.makedirs(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/GTEX/", exist_ok=True)
filtered_save.to_csv(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/GTEX/GTEX_BreastMammaryTissue.csv", index=False)


In [43]:
maf_thresh     = 0.05  # 最低次等位基因频率
abs_slope_min  = 0.5   # 最小绝对效应值
slope_se_max   = 0.1   # 最大效应标准误
p_nominal_max  = 1e-5  # 最大名义p值
p_permuted_max = 0.05  # 最大置换p值
fdr_thresh     = 0.05  # 最大 FDR

In [51]:
original_dataset_list = os.listdir(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/GTEX/")
original_dataset_list = [f for f in original_dataset_list if f.endswith(".txt") and not f.startswith("README")]
original_dataset_list

for dataset in original_dataset_list:
    dataset_path = f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/GTEX/{dataset}"
    dataset_name = dataset.split(".")[0]
    print(dataset_name)
    
    df = pd.read_csv(dataset_path,
                        sep='\t')
    df = df.join(epict_manifest, on="cpg_id", how="left")  # 左连接，缺失值为 NaN

    # 4. 使用 pyliftover 将 hg19 坐标转换到 hg38

    def liftover_pos(row):
        if pd.isna(row["chr_hg19"]) or pd.isna(row["pos_hg19"]):
            return pd.Series([None, None])
        chrom = f"chr{row['chr_hg19']}"
        # MAPINFO 是 1-based, pyliftover 也期望 1-based
        result = lo.convert_coordinate(chrom, int(row["pos_hg19"]))
        if not result:
            return pd.Series([None, None])
        # 取第一个匹配，result 返回 [(chrom, pos, strand, score), ...]
        new_chrom, new_pos, strand, score = result[0]
        return pd.Series([new_chrom.replace("chr", ""), new_pos])
    # 执行批量 liftover
    df[['chr_hg38', 'pos_hg38']] = df.apply(liftover_pos, axis=1)
    
    slope_se_max = 0.025
    df_len = 0
    df['SNP_region_start'] = df['variant_id'].str.split('_').str[1].astype(int)
    df['SNP_region_end'] = df['variant_id'].str.split('_').str[1].astype(int)
    df['SNP_ref'] = df['variant_id'].str.split('_').str[2]
    df['SNP_alt'] = df['variant_id'].str.split('_').str[3]
    # if df["slope_se"] <= slope_se_max  length less than 500, set slope_se_max to 0.1
    while df_len < 200:
        slope_se_max = slope_se_max * 2
        filtered = df[
            (df["maf"] >= maf_thresh) &
            (df["slope"].abs() >= abs_slope_min) &
            (df["slope_se"] <= slope_se_max) &
            (df["pval_nominal"] <= p_nominal_max) &
            (df["pval_permuted"] <= p_permuted_max) &
            (df["qval"] <= fdr_thresh) & 
            (df['SNP_ref'].str.len()==1) &
            (df['SNP_alt'].str.len()==1) &
            (df['SNP_region_start'] == df['SNP_region_start'])
        ].copy()
        df_len = len(filtered)

    print(f"{dataset_name} slope_se_max {slope_se_max} 筛选后保留 {len(filtered)} 条 mQTL 记录")
    
    filtered['SNP_region_start'] = filtered['variant_id'].str.split('_').str[1].astype(int)
    filtered['SNP_region_end'] = filtered['variant_id'].str.split('_').str[1].astype(int)
    filtered['SNP_ref'] = filtered['variant_id'].str.split('_').str[2]
    filtered['SNP_alt'] = filtered['variant_id'].str.split('_').str[3]
    filtered['chrom'] = filtered['variant_id'].str.split('_').str[0]
    filtered['CPG_region_start'] = filtered['pos_hg38']
    filtered['CPG_region_end'] = filtered['pos_hg38']
    filtered['effect_size'] = filtered['slope']
    filtered['se'] = filtered['slope_se']
    
    filtered_save = filtered.copy()
    filtered_save = filtered_save[['chrom', 'SNP_region_start', 'SNP_region_end', 'SNP_ref', 'SNP_alt', 'CPG_region_start', 'CPG_region_end', 'effect_size', 'se']]
    # distance less than 9000
    filtered_save = filtered_save[(filtered_save['SNP_region_start'] - filtered_save['CPG_region_start']).abs() < 9000]
    if not os.path.exists(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/GTEX/"):
        os.makedirs(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/GTEX/", exist_ok=True)
    filtered_save.to_csv(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/GTEX/GTEX_{dataset_name}.csv", index=False)

Ovary
Ovary slope_se_max 0.05 筛选后保留 1170 条 mQTL 记录
Prostate
Prostate slope_se_max 0.05 筛选后保留 272 条 mQTL 记录
MuscleSkeletal
MuscleSkeletal slope_se_max 0.1 筛选后保留 910 条 mQTL 记录
Lung
Lung slope_se_max 0.05 筛选后保留 3455 条 mQTL 记录
ColonTransverse
ColonTransverse slope_se_max 0.05 筛选后保留 4276 条 mQTL 记录
KidneyCortex
KidneyCortex slope_se_max 0.1 筛选后保留 1592 条 mQTL 记录
BreastMammaryTissue
BreastMammaryTissue slope_se_max 0.1 筛选后保留 1200 条 mQTL 记录
WholeBlood
WholeBlood slope_se_max 0.1 筛选后保留 1867 条 mQTL 记录
Testis
Testis slope_se_max 0.1 筛选后保留 2447 条 mQTL 记录
